In [0]:
# ── Cell 1: Authentication ────────────────────────────────────
storage        = "ukweatherstorage"
tenant         = "60e31c1d-fc60-4780-a4c9-9fd7e64047c9"

storage_key = dbutils.secrets.get(
    scope="uk-weather-kv-scope",
    key="storage-access-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage}.dfs.core.windows.net",
    storage_key
)

# ── Layer paths (Bronze still file-based) ─────────────────────
bronze = f"abfss://bronze@{storage}.dfs.core.windows.net/"
silver = f"abfss://silver@{storage}.dfs.core.windows.net/"
gold   = f"abfss://gold@{storage}.dfs.core.windows.net/"
models = f"abfss://models@{storage}.dfs.core.windows.net/"

# ── Unity Catalog namespaces ──────────────────────────────────
catalog        = "uk_weather_databricks"
silver_catalog = f"{catalog}.silver"
gold_catalog   = f"{catalog}.gold"
ml_catalog     = f"{catalog}.ml"

# ── Imports ───────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from datetime import datetime

# ── Utility functions ─────────────────────────────────────────
def log_header(table_name, layer="SILVER"):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print("="*60)
    print(f"  {layer} TRANSFORM — {table_name.upper()}")
    print(f"  Started: {ts}")
    print("="*60)

def log_footer(table_name, row_count, status, layer="SILVER"):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print("-"*60)
    print(f"  {layer} {table_name.upper()} — {status}")
    print(f"  Rows: {row_count:,}  |  Completed: {ts}")
    print("="*60)

def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except:
        return False

def list_bronze_json_paths(data_type):
    base_path = f"{bronze}{data_type}/"
    if not path_exists(base_path):
        raise Exception(f"Bronze path not found: {base_path}")
    print(f"  Scanning: {base_path}")
    file_count = 0
    for year in dbutils.fs.ls(base_path):
        for month in dbutils.fs.ls(year.path):
            for day in dbutils.fs.ls(month.path):
                for hour in dbutils.fs.ls(day.path):
                    for city in dbutils.fs.ls(hour.path):
                        files = [f for f in dbutils.fs.ls(city.path)
                                 if f.name.endswith(".json")]
                        file_count += len(files)
    print(f"  JSON files found: {file_count}")
    return f"{base_path}*/*/*/*/*/*.json"

def write_silver_table(df, table_name, partition_col="reading_date", mode="append"):
    """
    Writes Silver data to ADLS Gen2 and registers as
    external table in Unity Catalog.
    """
    adls_path  = f"{silver}{table_name}"
    full_table = f"{silver_catalog}.{table_name}"
    row_count  = df.count()

    print(f"  Writing {row_count:,} rows to ADLS Gen2...")
    print(f"  Path:  {adls_path}")

    df.write \
      .format("delta") \
      .mode(mode) \
      .option("mergeSchema", "true") \
      .partitionBy(partition_col) \
      .save(adls_path)

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {full_table}
        USING DELTA
        LOCATION '{adls_path}'
    """)

    print(f"  Registered in Unity Catalog: {full_table}")
    print(f"  ADLS Gen2 owns data at:      {adls_path}")
    return row_count

def write_gold_table(df, table_name, partition_col="reading_date", mode="overwrite"):
    adls_path  = f"{gold}{table_name}"
    full_table = f"{gold_catalog}.{table_name}"
    row_count  = df.count()

    print(f"  Writing {row_count:,} rows to ADLS Gen2...")
    print(f"  Path:  {adls_path}")

    df.write \
      .format("delta") \
      .mode(mode) \
      .option("overwriteSchema", "true") \
      .partitionBy(partition_col) \
      .save(adls_path)

    # Drop and recreate to handle schema changes cleanly
    spark.sql(f"DROP TABLE IF EXISTS {full_table}")
    spark.sql(f"""
        CREATE TABLE {full_table}
        USING DELTA
        LOCATION '{adls_path}'
    """)

    print(f"  Registered in Unity Catalog: {full_table}")
    print(f"  ADLS Gen2 owns data at:      {adls_path}")
    return row_count
    
def verify_unity_table(table_name, primary_key=None):
    full_table = f"{silver_catalog}.{table_name}"
    df         = spark.table(full_table)
    total      = df.count()
    status     = "PASS" if total > 0 else "FAIL"
    print(f"  Table:  {full_table}")
    print(f"  Rows:   {total:,} | {status}")
    return total, 0, 0, status

print("="*60)
print("  Utils loaded successfully")
print(f"  Bronze:  {bronze}")
print(f"  Catalog: {catalog}")
print(f"  Silver:  {silver_catalog}")
print(f"  Gold:    {gold_catalog}")
print(f"  ML:      {ml_catalog}")
print("="*60)